# Assignment 4: Python/pandas translation

This notebook translates the R summary in `polyglot-parity.qmd` into Python/pandas:
a descriptive, **unweighted** summary of BMI by `IncomeGroup` and `Gender`.

**Run order:** run every cell here first (it saves two small files to `output/`),
then render `polyglot-parity.qmd`, which reads those files to build the parity table.
Before committing, clear outputs with *Clear All Outputs* (or
`jupyter nbconvert --clear-output --inplace translation.ipynb`).

## 1. Setup: packages, helper, and a relative data path

In [ ]:
import pandas as pd

from helpers import find_repo_root, summarise_mean_sd

repo_root = find_repo_root()
data_path = repo_root / "examples" / "nhanes-equity" / "data" / "nhanes_equity_v6.csv"
out_dir = repo_root / "assignments" / "assignment04-polyglot" / "submission" / "output"
out_dir.mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__)

## 2. Load and inspect  (R: `read_csv()`, `nrow()`, `ncol()`, `names()`)

In [ ]:
# low_memory=False reads the whole file at once, so pandas does not guess
# types chunk by chunk (without it, pandas warns that Marital has mixed types)
nhanes = pd.read_csv(data_path, low_memory=False)

raw_rows, raw_cols = nhanes.shape
print("rows:", raw_rows, "| columns:", raw_cols)
print(nhanes.columns.tolist())

## 3. Missingness filter  (R: `filter(!is.na(BMI), !is.na(IncomeGroup), !is.na(Gender))`)

In [ ]:
analysis = nhanes.dropna(subset=["BMI", "IncomeGroup", "Gender"])

filtered_rows = len(analysis)
print("rows after filter:", filtered_rows, "| dropped:", raw_rows - filtered_rows)
nhanes[["BMI", "IncomeGroup", "Gender"]].isna().sum()

## 4. Step-by-step translation of the R pipeline

| R (`dplyr`) | Python (`pandas`) |
|---|---|
| `group_by(IncomeGroup, Gender)` | `.groupby(["IncomeGroup", "Gender"], as_index=False)` |
| `summarise(n = n(), mean_bmi = mean(BMI), sd_bmi = sd(BMI))` | `.agg(n=("BMI", "size"), mean_bmi=("BMI", "mean"), sd_bmi=("BMI", "std"))` |
| `round(x, 2)` | `.round(2)` |
| `arrange(IncomeGroup, Gender)` | `.sort_values(["IncomeGroup", "Gender"])` |

In [ ]:
py_summary = (
    analysis
    .groupby(["IncomeGroup", "Gender"], as_index=False)
    .agg(
        n=("BMI", "size"),
        mean_bmi=("BMI", "mean"),
        sd_bmi=("BMI", "std"),
    )
    .sort_values(["IncomeGroup", "Gender"])
    .reset_index(drop=True)
)
py_summary["mean_bmi"] = py_summary["mean_bmi"].round(2)
py_summary["sd_bmi"] = py_summary["sd_bmi"].round(2)

py_summary

## 5. Same result through the reusable helper  (R: `summarise_mean_sd()`)

In [ ]:
py_function_summary = summarise_mean_sd(nhanes, "BMI", "IncomeGroup", "Gender")

# The helper must reproduce the step-by-step pipeline exactly
same = py_function_summary.rename(
    columns={"mean_value": "mean_bmi", "sd_value": "sd_bmi"}
).equals(py_summary)
print("helper matches step-by-step pipeline:", same)
py_function_summary

## 6. Save small results for the parity table in `polyglot-parity.qmd`

In [ ]:
py_summary.to_csv(out_dir / "python-summary.csv", index=False)

pd.DataFrame({
    "metric": ["raw_row_count", "raw_column_count", "filtered_row_count"],
    "python_value": [raw_rows, raw_cols, filtered_rows],
}).to_csv(out_dir / "python-counts.csv", index=False)

print("saved to", out_dir.relative_to(repo_root))